# 05 — Baseline LogReg + TF-IDF · Databricks
**Exécuter après** : `04_feature_engineering_db.ipynb`
---

In [ ]:
%pip install scipy scikit-learn joblib seaborn -q

In [ ]:
import sys, os, json, time
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import load_npz

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, roc_curve)

plt.style.use('dark_background')

COLAB_DIR   = '/dbfs/FileStore/fakenews/colab_exports'
MODELS_DIR  = '/dbfs/FileStore/fakenews/models'
REPORTS_DIR = '/dbfs/FileStore/fakenews/reports'
for d in [MODELS_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)
print('Imports OK')

## Section 1 — Chargement des features

In [ ]:
X_train = load_npz(os.path.join(COLAB_DIR, 'train_features.npz'))
X_test  = load_npz(os.path.join(COLAB_DIR, 'test_features.npz'))
y_train = np.load(os.path.join(COLAB_DIR, 'train_labels.npy'))
y_test  = np.load(os.path.join(COLAB_DIR, 'test_labels.npy'))

print(f'Train : {X_train.shape} | labels : {y_train.shape}')
print(f'Test  : {X_test.shape}  | labels : {y_test.shape}')
print(f'Distribution train — FAKE: {y_train.sum():,} | REAL: {(y_train==0).sum():,}')

## Section 2 — Entraînement LogReg

In [ ]:
print('Entrainement LogisticRegression...')
start = time.time()
clf = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced',
                          solver='lbfgs', n_jobs=-1, random_state=42)
clf.fit(X_train, y_train)
print(f'Entrainement termine en {time.time() - start:.1f}s')

## Section 3 — Évaluation

In [ ]:
y_pred  = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
f1       = f1_score(y_test, y_pred, average='weighted')
auc      = roc_auc_score(y_test, y_proba)

print(f'=== METRIQUES BASELINE ===')
print(f'  Accuracy : {accuracy:.4f}')
print(f'  F1-score : {f1:.4f}')
print(f'  AUC-ROC  : {auc:.4f}')
print(f'\n{classification_report(y_test, y_pred, target_names=["REAL", "FAKE"])}')

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['REAL', 'FAKE'], yticklabels=['REAL', 'FAKE'])
ax1.set_title('Matrice de confusion — Baseline')

fpr, tpr, _ = roc_curve(y_test, y_proba)
ax2.plot(fpr, tpr, color='#00ccaa', linewidth=2, label=f'AUC = {auc:.3f}')
ax2.plot([0, 1], [0, 1], 'gray', linestyle='--')
ax2.set_title('Courbe ROC — Baseline')
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'baseline_roc.png'), dpi=150, bbox_inches='tight')
plt.show()

## Section 4 — Sauvegarde

In [ ]:
baseline_dir = os.path.join(MODELS_DIR, 'baseline')
os.makedirs(baseline_dir, exist_ok=True)
model_path = os.path.join(baseline_dir, 'tfidf_logreg.pkl')
joblib.dump(clf, model_path)
print(f'Modele sauvegarde : {model_path}')

metadata_path = os.path.join(MODELS_DIR, 'metadata.json')
metadata = {}
if os.path.exists(metadata_path):
    with open(metadata_path) as f:
        metadata = json.load(f)

metadata['baseline_logreg'] = {
    'accuracy': round(accuracy, 4), 'f1_score': round(f1, 4),
    'auc_roc': round(auc, 4), 'trained_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'model_path': model_path,
}
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Metadata mis a jour : {metadata_path}')
print('Prochaine etape : 06_nlp_classical_db.ipynb')